# More on PySpark: join and others

In [0]:
from pyspark.sql import functions as F

In [0]:
# name your database
db = "dsc592.retail"

- `.columns`: print column names.
- `.printSchema()`: prints column names, types and nullability, as a tree

Define dataframes: these are python variables

In [0]:
f = spark.table(f"{db}.fact_sales")
c  = spark.table(f"{db}.dim_customer")
p  = spark.table(f"{db}.dim_product")
d = spark.table(f"{db}.dim_date")

## More on Joining syntax
1. `A.join(B, A.key = B.key)`: we used this in L9spark.ipynb
2. `A.join(B, on = "key")`: This is preferred since the 1st generates 2 "key" columns.
3. `A.join(B, F.col("tableA.key") == F.col("tableB.key"))`: can also use an expression

In [0]:
# from L9spark.ipynb: joining sales with dates
# how come it didn't matter in L9?

# joining using 2.


In [0]:
# joining using item 3
f.join(d, F.col("f.date_key") == F.col("d.date_key")).show(3)

## extra conditions with join
If we want to use extra condition(s) along with joining, we often use the joining syntax that involves `==`

In [0]:
f.join(c, (f.customer_id == c.customer_id) & (c.segment == "SMB"), "left").show(3)

## Chaining joins

In [0]:
enriched = (f
  .join(c, on="customer_id", how="left")      # keep sales with unknown customers
  .join(p, on="product_id",  how="inner")
  .select("order_id", "date_key", "customer_id", "name", "segment",
          "category", "quantity", "amount"))


## left anti
- `A.join(B, on = "key", how = "left_anti")`: rows in A with no match in B. **can be good for finding new records**

In [0]:
# left anti: “rows in A with no match in B”
# will only return rows from A
nc = spark.table(f"{db}.stg_customer")
(nc.join(c, on = "customer_id", how = "left_anti")
   .orderBy("customer_id")
   .show(3))